# Версия 3, част 1 — коя е тази единица? (crop класификатор)

**Защо започнахме наново.** Версия 2 гледаше целия screenshot (1280×720) и трябваше да каже кои от 48 единици
има. Една единица заема около 50×60 пиксела — под 0.5% от картинката. Моделът трябваше едновременно да
*намери* малките фигурки и да ги *разпознае*, а освен това не можеше да брои: „двама Footman“ и „един
Footman“ дават един и същ етикет.

**Новият план** разделя задачата на две прости части:

1. **Къде има единици?** — детектор (`warcraft3_detector.ipynb`), една рамка за всяка единица.
2. **Коя е тази единица?** — този notebook: класификатор, който гледа *само* изрязаната рамка и избира
   1 от 48 (softmax), точно като Lesson 1 / Версия 1, но с изображения от самата игра.

Crop-овете идват от solo снимките, смесените сцени (с позиции в `data/units.csv`) и проверените истински
screenshot-и. За solo и смесените сцени знаем мястото на всяка единица спрямо камерата и го пресмятаме —
без ръчно маркиране. Работи локално и в Kaggle (GPU, Internet включен, прикачен dataset).

In [ ]:
from fastai.vision.all import *

# where the data is (read) and where results go (write): the project folder, or Kaggle's folders
if Path('/kaggle/input').is_dir():
    ROOT = next(Path('/kaggle/input').rglob('single_units/labels.csv')).parents[2]
    WORK = Path('/kaggle/working')
else:
    ROOT = WORK = Path('.')
CROPS = WORK/'unit_crops'

## Къде е всяка единица на екрана

`dataset_trigger_single.j` и `dataset_trigger.j` държат камерата под ъгъл 304° (гледа надолу под 56°), на
разстояние `zoom`, над точка (cx, cy). Мястото на единицата спрямо тази точка (x, y) става пиксели с
обикновена перспективна проекция. Трите числа `F`, `CX`, `CY` са измерени от няколко снимки
(F = фокусно разстояние в пиксели, CX/CY = къде на екрана се вижда точката на камерата).
Рамката е по-висока нагоре, за да хване и летящите единици.

In [ ]:
F, CX, CY = 908, 630, 286
COS, SIN = np.cos(np.radians(56)), np.sin(np.radians(56))

def screen_box(x, y, zoom):
    "(left, top, right, bottom) in pixels of a 1280x720 screenshot"
    depth = zoom + y * COS
    sx, sy, scale = CX + F * x / depth, CY - F * y * SIN / depth, F / depth
    return (sx - 175*scale, sy - 320*scale, sx + 175*scale, sy + 145*scale)

def visible(box):
    "Is the unit on the battlefield (not off screen, not under the bottom panel)?"
    sx, sy = (box[0] + box[2]) / 2, box[3] - 145 * (box[2] - box[0]) / 350
    return 0 <= sx <= 1280 and 30 <= sy <= 720 and not (sy > 540 and 140 < sx < 1140)

def solo_positions(r):
    offsetx = (r.position % 3 - 1) * 220.0          # same formulas as in dataset_trigger_single.j
    offsety = (r.position // 3 - 1) * 140.0 - 70.0
    return [(offsetx + (i - (r.copies - 1) / 2) * 340, offsety) for i in range(r.copies)]

def split_of(map_name):
    "Maps named ...valid... / ...test... are never trained on; ...trial... maps (short test runs) are not used."
    return 'trial' if 'trial' in map_name else 'valid' if 'valid' in map_name else 'test' if 'test' in map_name else 'train'

## Всички единици в една таблица

Един ред = една единица: снимка, split, име и рамка. Map 2 от solo снимките има кадри, изключени заради
текста „No clear capture area“ (y ≈ 415–431): от тях пазим само рамките, които не стигат до текста.

In [ ]:
WARNING_TEXT = (223, 405, 571, 441)
def overlaps(a, b): return a[0] < b[2] and b[0] < a[2] and a[1] < b[3] and b[1] < a[3]

solo = pd.read_csv(ROOT/'data/single_units/labels.csv')
excluded = pd.read_csv(ROOT/'data/single_units/excluded.csv')
warning = set(excluded.loc[excluded['reason'].str.contains('warning text'), 'fname'])
solo = solo[solo['split'].isin(['train', 'valid', 'test']) & ~solo['fname'].isin(set(excluded['fname']) - warning)]
vocab = sorted(solo['labels'].unique())

units = []
for r in solo.itertuples():
    for x, y in solo_positions(r):
        box = screen_box(x, y, r.zoom)
        if r.fname in warning and overlaps(box, WARNING_TEXT): continue
        units.append((ROOT/'data/single_units/screens'/r.fname, r.split, 'solo', r.labels, *box))

if (ROOT/'data/units.csv').is_file():
    for r in pd.read_csv(ROOT/'data/units.csv').itertuples():
        box = screen_box(r.x, r.y, r.zoom)
        if visible(box) and split_of(r.fname.rsplit('_', 1)[0]) != 'trial':
            units.append((ROOT/'data/screens'/r.fname, split_of(r.fname.rsplit('_', 1)[0]), 'mixed', r.label, *box))

for split in ['train', 'test']:
    folder = ROOT/'my_screenshots'/split
    for txt in sorted((folder/'labels').glob('*.txt')) if (folder/'labels').is_dir() else []:
        path = next(p for p in folder.iterdir() if p.stem == txt.stem and p.suffix.lower() in ('.png', '.jpg', '.jpeg'))
        w, h = Image.open(path).size
        for line in txt.read_text().splitlines():
            c, x, y, bw, bh = [float(v) for v in line.split()]
            units.append((path, split, 'real', vocab[int(c)], (x - bw/2) * w, (y - bh/2) * h, (x + bw/2) * w, (y + bh/2) * h))

units = pd.DataFrame(units, columns=['image', 'split', 'source', 'label', 'left', 'top', 'right', 'bottom'])
units.groupby(['source', 'split']).size()

## Изрязване

Всяка единица става отделна малка снимка. В смесените сцени в рамката понякога влизат и съседни единици —
класификаторът трябва да назове тази в средата, както ще е и в истинска игра.

In [ ]:
(CROPS/'images').mkdir(parents=True, exist_ok=True)
names = []
for path, rows in units.groupby('image', sort=False):
    img = None
    for i, u in enumerate(rows.itertuples()):
        name = f'{u.split}_{u.source}_{Path(path).stem}_{i}.jpg'
        if not (CROPS/'images'/name).exists():
            if img is None: img = Image.open(path).convert('RGB')
            img.crop((round(u.left), round(u.top), round(u.right), round(u.bottom))).save(CROPS/'images'/name, quality=95)
        names.append((u.Index, name))
units.loc[[i for i, _ in names], 'fname'] = [n for _, n in names]
units.to_csv(CROPS/'crops.csv', index=False)
units.groupby('split')['label'].value_counts().groupby('split').agg(['min', 'max', 'count'])

## DataBlock и аугментация

- `CategoryBlock` — точно една единица на crop, softmax, както Версия 1.
- Validation = цели карти, които моделът не е виждал, не случайни кадри.
- Аугментация, подходяща за играта: огледален образ (единиците гледат във всички посоки), лек zoom и
  отместване (детекторът няма да реже точно като нашата формула), промяна на светлината (ден/нощ).
  Без голямо въртене — камерата в Warcraft III винаги е под същия ъгъл.

In [ ]:
df = units[units['split'].isin(['train', 'valid'])].copy()
df['is_valid'] = df['split'].eq('valid')
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock(vocab=vocab)),
    get_x=ColReader('fname', pref=CROPS/'images'), get_y=ColReader('label'),
    splitter=ColSplitter('is_valid'),
    item_tfms=Resize(224, method='squish'),
    batch_tfms=aug_transforms(max_rotate=5, max_zoom=1.25, max_lighting=0.3, max_warp=0)
).dataloaders(df, bs=32, num_workers=0 if os.name == 'nt' else 4)
dls.show_batch(max_n=12, nrows=2)

## Обучение

`convnext_tiny.fb_in22k` — същият модел, който беше най-добър във Версия 1. `fine_tune`: първо само новата
глава, после целият модел. На GTX 1080 Ti обучението понякога внезапно дава `nan` в случаен batch, затова,
както във Версия 2: `SaveModelCallback` пази най-добрата епоха по `valid_loss`, а `TerminateOnNaNCallback`
спира, ако се появи `nan`. `to_fp16()` (mixed precision) прави обучението по-бързо и с по-малко памет.

In [ ]:
set_seed(42)
learn = vision_learner(dls, 'convnext_tiny.fb_in22k', metrics=[accuracy, error_rate], path=CROPS).to_fp16()
learn.fine_tune(8, base_lr=2e-3, cbs=[SaveModelCallback(monitor='valid_loss', fname='best'),
                                     TerminateOnNaNCallback()])
learn.load('best')
learn.validate()

## Резултати на непознатите карти

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.most_confused(min_val=2)

In [ ]:
interp.plot_top_losses(9, nrows=3)

In [ ]:
# test maps and real test screenshots: looked at once, after all choices are made
test = units[units['split'].eq('test')]
for source, rows in test.groupby('source'):
    _, acc, _ = learn.validate(dl=dls.test_dl(rows, with_labels=True))
    print(f'test {source}: {acc:.1%} of {len(rows)} crops')

In [ ]:
learn.export('unit_classifier.pkl')
# on Kaggle everything in /kaggle/working becomes the notebook's output: keep the model, not the crops
if WORK != ROOT: shutil.rmtree(CROPS/'images'); (CROPS/'models'/'best.pth').unlink()